In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textblob import TextBlob
from deep_translator import GoogleTranslator
import numpy as np

In [2]:
## Read the billing dataset
billing = pd.read_csv('billing.csv', sep='#')

##Remaning the columns
month_map = {'Janvier': 'January', 'Février': 'February', 'Mars': 'March', 'Avril': 'April', 'Mai': 'May', 'Juin': 'June', 'Juillet': 'July', 'Août': 'August', 'Septembre': 'September', 'Octobre': 'October', 'Novembre': 'November', 'Décembre': 'December'}
billing['Month'] = billing['mois'].map(month_map)
billing.rename(columns={
    'identifiant_abonne': 'Customer_ID',
    'Montant facturé': 'Amount_billed'
}, inplace=True)

## drop the mois column
billing = billing.drop(columns=['mois'])

## Rearranging the columns
new_column_order = ['Customer_ID','Month', 'Amount_billed'] ##rearranging the columns
billing = billing[new_column_order]

## Pivot the dataset
billing_pivot = billing.pivot_table(index="Customer_ID", columns="Month", values="Amount_billed", aggfunc="sum").reset_index()

## Defining Months in chronological order
month_order = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]

## Ensuring months are in chronological order
billing_pivot = billing_pivot[["Customer_ID"] + month_order]

## Function to treat the missing values
def custom_fillna(usage_data):
    def process_row(row):
        ## Flag to check if we've encountered a valid number
        found_value = False
        ## This will help to skip the column 'Customer_ID'
        for col in usage_data.columns[1:]:
            ## If the value is missing
            if pd.isna(row[col]):
                if not found_value:
                    row[col] = np.nan  ## If blanks are at the start keep them as NaN
                else:
                    row[col] = 0  ## If blanks are in between then fill it with 0
            else:
                found_value = True  ## if the first valid number is found, so future blanks get 0
        return row

    return usage_data.apply(process_row, axis=1)

# Apply missing value treatment to both DataFrames
billing_pivot = custom_fillna(billing_pivot)

## Print the result
billing_pivot

Month,Customer_ID,January,February,March,April,May,June,July,August,September,October,November,December
0,SUB-100962022,16.65,44.22,180.29,48.77,22.31,42.86,9.99,11.14,28.97,12.38,64.58,20.44
1,SUB-101759923,12.91,39.12,39.02,10.44,58.79,14.80,105.99,48.03,20.32,130.83,18.58,22.19
2,SUB-104250558,78.21,110.26,5.61,82.70,20.73,20.28,37.69,28.21,136.23,19.02,12.42,47.61
3,SUB-104899978,83.00,192.48,1.08,0.18,36.11,0.18,104.64,40.98,60.34,42.38,17.52,48.43
4,SUB-105344122,36.15,168.97,30.41,21.46,70.56,32.27,23.58,89.60,29.95,55.42,34.57,109.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,SUB-996400695,NaN,135.77,35.41,27.23,64.95,42.53,68.15,64.88,82.79,126.78,17.74,38.38
996,SUB-997894279,55.40,39.82,124.92,14.47,51.27,48.68,81.59,31.75,43.08,19.78,30.42,97.94
997,SUB-998218459,83.48,18.21,17.49,19.68,162.05,10.59,56.74,9.99,49.99,22.44,20.79,39.19
998,SUB-998702037,92.44,19.13,21.14,19.34,24.80,4.78,72.06,76.86,63.83,44.32,16.60,238.44


In [16]:
## Exporting the updated dataset as an excel file
billing_pivot.to_excel('billing_updated.xlsx', index=False)

In [3]:
## Reading the subscribers tarrif dataset
subscribers_tariff = pd.read_csv('subscribers_tariff.csv', sep =';', header=None)

##Defining columns for the dataset as they were missing
subscribers_tariff.columns = ['Customer_ID', 'Plan']

## Create a dictionary for mapping the translation (Alternative method - Translate Manually)
translation_dict = {"Payez-au-fur-et-à-mesure": "Pay-as-you-go", "Tarif réduit": "Reduced rate"}

## Replace values in the 'Plan' column
subscribers_tariff['Plan'] = subscribers_tariff['Plan'].replace(translation_dict)
subscribers_tariff

## Add "SUB-" prefix to Customer_ID
subscribers_tariff['Customer_ID'] = "SUB-" + subscribers_tariff['Customer_ID'].astype(str)

## Print the result
subscribers_tariff


,Customer_ID,Plan
0,SUB-909383560,Pay-as-you-go
1,SUB-739952735,Reduced rate
2,SUB-518704321,Pay-as-you-go
3,SUB-508589057,Pay-as-you-go
4,SUB-555622338,Pay-as-you-go
...,...,...
995,SUB-592358144,Reduced rate
996,SUB-277070219,Reduced rate
997,SUB-405826209,Reduced rate
998,SUB-475645336,Pay-as-you-go


In [122]:
# Exporting the updated dataset as an excel file
subscribers_tariff.to_excel('subscribers_tariff_updated.xlsx', index=False)

In [4]:
## Reading the usage dataset which is in excel format
usage_data = pd.read_excel('usage_data.xlsx')

##load all the sheets present in the excel sheet
sheet_names=['utilisation_donnees_kilo_octets', 'temps_appel_secondes']

## Define the sheet into different dataframes
Data = pd.read_excel('usage_data.xlsx', sheet_name=sheet_names[0])
talktime = pd.read_excel('usage_data.xlsx', sheet_name=sheet_names[1])

## Reanaming the columns similar to the other datasets for both the sheets
new_column_names = {"sid": "Customer_ID","Janvier": "January","Février": "February","Mars": "March","Avril": "April", "Mai": "May", "Juin": "June","Juillet": "July","Août": "August","Septembre": "September","Octobre": "October","Novembre": "November", "Décembre": "December"}

Data.rename(columns=new_column_names, inplace=True)
talktime.rename(columns=new_column_names, inplace=True)

## Function to treat the missing values
def custom_fillna(usage_data):
    def process_row(row):
        ## Flag to check if we've encountered a valid number
        found_value = False
        ## This will help to skip the column 'Customer_ID'
        for col in usage_data.columns[1:]:
            ## If the value is missing
            if pd.isna(row[col]):
                if not found_value:
                    row[col] = np.nan  ## If blanks are at the start keep them as NaN
                else:
                    row[col] = 0  ## If blanks are in between then fill it with 0
            else:
                found_value = True  ## if the first valid number is found, so future blanks get 0
        return row

    return usage_data.apply(process_row, axis=1)

# Apply missing value treatment to both DataFrames
Data = custom_fillna(Data)
talktime = custom_fillna(talktime)

## Convert KB to MB in Data (1 MB = 1024 KB)
for col in Data.columns[1:]:  # Skip 'Customer_ID'
    Data[col] = Data[col] / 1024

## Convert seconds to minutes in talktime (1 min = 60 sec)
for col in talktime.columns[1:]:  # Skip 'Customer_ID'
    talktime[col] = talktime[col] / 60

In [6]:
## Print the result
Data
talktime

,Customer_ID,January,February,March,April,May,June,July,August,September,October,November,December
0,SUB-100962022,0.655520,0.000000,13.965840,0.000000,4.850733,0.771581,0.000000,3.567566,1.167214,10.912336,10.455371,7.479034
1,SUB-101759923,1.518176,0.000000,1.710406,8.672314,9.952589,2.876901,1.387716,2.713978,4.737218,7.344614,2.089351,0.192969
2,SUB-104250558,6.336714,1.819621,8.607627,17.709554,0.934999,0.000000,4.537011,0.438685,6.537872,1.062470,0.000000,14.492869
3,SUB-104899978,3.782543,0.000000,11.149644,1.904361,0.190911,1.014306,0.000000,5.876778,7.836092,3.073795,1.416548,2.271919
4,SUB-105344122,0.605812,2.757354,8.760395,3.946631,0.000000,4.037238,1.662711,0.000000,8.068125,3.314118,10.880927,7.981499
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,SUB-996400695,0.000000,0.853336,14.725982,0.965562,4.096914,0.935322,6.297421,0.000000,14.425496,1.331667,0.000000,0.000000
996,SUB-997894279,4.665053,0.431564,8.141613,4.520940,1.938474,12.357150,2.395868,1.738567,4.204622,0.097698,7.057962,8.446960
997,SUB-998218459,9.919606,3.477921,1.447680,3.499486,6.816858,7.333542,2.689875,0.000000,2.350675,2.081930,8.309178,9.087822
998,SUB-998702037,7.467846,4.467165,0.000000,1.557508,3.579778,7.552106,5.089358,0.000000,4.237873,5.500864,1.558046,3.052855


In [117]:
# ## Exporting the updated dataset as an excel file
output_file = "usage_data_updated.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer: ## Creates an Excel writer object using openpyxl, which allows writing to .xlsx files.
    Data.to_excel(writer, sheet_name=sheet_names[0], index=False)
    talktime.to_excel(writer, sheet_name=sheet_names[1], index=False)

In [7]:
# Load the Excel sheets
data = pd.read_excel("usage_data_updated.xlsx", sheet_name="utilisation_donnees_kilo_octets", engine="openpyxl")
talktime = pd.read_excel("usage_data_updated.xlsx", sheet_name="temps_appel_secondes", engine="openpyxl")
tariff = pd.read_excel("subscribers_tariff_updated.xlsx", engine="openpyxl")
billing = pd.read_excel("billing_updated.xlsx", engine="openpyxl")

# Step 1: Merge data usage and talktime
merged_data = pd.merge(data, talktime, on="Customer_ID", how="inner", suffixes=("_Data", "_Talktime"))

# Step 2: Add tariff information
merged_data = pd.merge(merged_data, tariff, on="Customer_ID", how="inner")

# Step 3: Merge billing details
final_merged_dataset = pd.merge(merged_data, billing, on="Customer_ID", how="inner")

# Step 4: Exclude non-numeric columns for statistical analysis
numeric_merged_dataset = final_merged_dataset.select_dtypes(include=['number'])

# Step 5: Calculate summary statistics
summary_stats = {
    "Sum": numeric_merged_dataset.sum(),
    "Mean": numeric_merged_dataset.mean(),
    "Median": numeric_merged_dataset.median(),
    "Mode": numeric_merged_dataset.mode().iloc[0]  # First mode value for each column
}

In [16]:
final_merged_dataset

,Customer_ID,January_Data,February_Data,March_Data,April_Data,May_Data,June_Data,July_Data,August_Data,September_Data,...,March,April,May,June,July,August,September,October,November,December
0,SUB-100962022,660.462057,3422.927240,16959.662252,3877.704846,1206.483719,3281.140686,0.000000,94.865692,1887.848948,...,180.29,48.77,22.31,42.86,9.99,11.14,28.97,12.38,64.58,20.44
1,SUB-101759923,281.312306,2912.786868,2892.606619,0.000000,4829.669299,465.266220,9589.530458,3788.511218,1007.342189,...,39.02,10.44,58.79,14.80,105.99,48.03,20.32,130.83,18.58,22.19
2,SUB-104250558,3878.502731,5503.845638,239.075969,4053.650627,1031.661917,1013.471285,1861.509230,1405.867685,6779.302394,...,5.61,82.70,20.73,20.28,37.69,28.21,136.23,19.02,12.42,47.61
3,SUB-104899978,4131.776639,9623.947243,0.000000,0.000000,1800.178007,0.000000,5231.109065,2021.925456,2980.526925,...,1.08,0.18,36.11,0.18,104.64,40.98,60.34,42.38,17.52,48.43
4,SUB-105344122,1802.835047,8434.815250,1479.471834,1054.917919,3527.024881,1590.910894,1169.503298,4479.586336,1456.684630,...,30.41,21.46,70.56,32.27,23.58,89.60,29.95,55.42,34.57,109.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,SUB-996400695,0.000000,6783.876759,1702.100025,1356.209849,3224.331248,2121.502537,3375.428314,3243.748966,4071.138843,...,35.41,27.23,64.95,42.53,68.15,64.88,82.79,126.78,17.74,38.38
996,SUB-997894279,4515.631816,2977.962789,11447.433947,422.434887,4117.107147,3803.018509,7144.083222,2165.170816,3283.416100,...,124.92,14.47,51.27,48.68,81.59,31.75,43.08,19.78,30.42,97.94
997,SUB-998218459,7298.549774,801.732892,739.094184,948.819229,15170.783033,19.562806,4659.730039,0.000000,3984.489926,...,17.49,19.68,162.05,10.59,56.74,9.99,49.99,22.44,20.79,39.19
998,SUB-998702037,4585.823517,933.670638,1056.744183,957.384610,1221.686542,202.095605,3575.424157,3842.192984,3168.218272,...,21.14,19.34,24.80,4.78,72.06,76.86,63.83,44.32,16.60,238.44


In [15]:
print(pd.DataFrame(summary_stats))

                            Sum         Mean       Median   Mode
January_Data       3.359857e+06  3547.895031  2482.717837   0.00
February_Data      3.319322e+06  3404.432913  2413.872143   0.00
March_Data         3.509845e+06  3552.474358  2527.545529   0.00
April_Data         3.578602e+06  3578.602291  2551.107614   0.00
May_Data           3.512152e+06  3512.152199  2423.292537   0.00
June_Data          3.468966e+06  3468.965748  2438.204419   0.00
July_Data          3.445418e+06  3445.418155  2424.248009   0.00
August_Data        3.641949e+06  3641.949281  2510.401807   0.00
September_Data     3.549626e+06  3549.625752  2610.408201   0.00
October_Data       3.741595e+06  3741.594534  2499.797901   0.00
November_Data      3.316514e+06  3316.513667  2355.416067   0.00
December_Data      3.570966e+06  3570.966388  2529.765556   0.00
January_Billing    4.980518e+04    51.611585    39.830000   9.99
February_Billing   4.989627e+04    50.759176    36.960000   9.99
March_Billing      5.2563

In [17]:
# Export the combined dataset
final_merged_dataset.to_csv("Combined_Data.csv", index=False)